In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#Loading data and handling missing values

In [ ]:
import pandas as pd
import numpy as np

# Load data
train = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")

# List of datasets for consistent cleaning
datasets = [train, test]

# Categorical features where NA means "None"
none_fill = [
    "PoolQC","MiscFeature","Alley","Fence","FireplaceQu",
    "GarageType","GarageFinish","GarageQual","GarageCond",
    "BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2",
    "MasVnrType"
]
for df in datasets:
    df[none_fill] = df[none_fill].fillna("None")

# Numerical features where NA means 0
zero_fill = [
    "GarageYrBlt","GarageArea","GarageCars","BsmtFinSF1","BsmtFinSF2",
    "BsmtUnfSF","TotalBsmtSF","BsmtFullBath","BsmtHalfBath","MasVnrArea"
]
for df in datasets:
    df[zero_fill] = df[zero_fill].fillna(0)

# LotFrontage → median by neighborhood
for df in datasets:
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )

# Remaining categorical → mode
for df in datasets:
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].fillna(df[col].mode()[0])


In [ ]:
import matplotlib.pyplot as plt

# Example: Detect outliers in GrLivArea vs SalePrice
plt.scatter(train["GrLivArea"], train["SalePrice"])
plt.xlabel("GrLivArea")
plt.ylabel("SalePrice")
plt.show()

# Drop extreme outliers
train = train.drop(train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)].index)


In [ ]:
#EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------
# 1. Load dataset
# -----------------------
train = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")

# -----------------------
# 2. Histogram of SalePrice
# -----------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.histplot(train['SalePrice'], bins=50, kde=True, color='skyblue')
plt.title("Histogram of SalePrice (Raw)")
plt.xlabel("SalePrice")

plt.subplot(1,2,2)
sns.histplot(np.log1p(train['SalePrice']), bins=50, kde=True, color='salmon')
plt.title("Histogram of SalePrice (Log-Transformed)")
plt.xlabel("Log(SalePrice)")

plt.tight_layout()
plt.show()

# -----------------------
# 3. Correlation heatmap
# -----------------------
plt.figure(figsize=(16,12))
numeric_features = train.select_dtypes(include=[np.number])
corr_matrix = numeric_features.corr()

# Optional: show only top correlations with SalePrice
top_corr = corr_matrix['SalePrice'].sort_values(ascending=False).head(15)
print("Top correlations with SalePrice:\n", top_corr)




In [ ]:
#Boxplot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
train = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")

plt.figure(figsize=(20,8))
sns.boxplot(x='Neighborhood', y='SalePrice', data=train)
plt.xticks(rotation=45, ha='right')
plt.title("SalePrice Distribution by Neighborhood")
plt.xlabel("Neighborhood")
plt.ylabel("SalePrice")
plt.show()


In [ ]:
#Heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
train = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")

# Select numeric features
numeric_features = train.select_dtypes(include=[np.number])

# Compute correlation matrix
corr_matrix = numeric_features.corr()

# Select top 10 features most correlated with SalePrice
top_features = corr_matrix['SalePrice'].abs().sort_values(ascending=False).iloc[1:11].index.tolist()

# Include SalePrice itself
top_features.append('SalePrice')

# Subset correlation matrix
top_corr_matrix = corr_matrix.loc[top_features, top_features]

# Plot heatmap
plt.figure(figsize=(12,10))
sns.heatmap(top_corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Top 10 Features with SalePrice")
plt.show()


In [ ]:
#Splitting the data into training and testing 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# Load dataset
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")

# Features and target
X = train_df.drop("SalePrice", axis=1)
y = train_df["SalePrice"]

# Optional: log-transform target for regression stability
y = np.log1p(y)

# 80-20 train-test split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check shapes
print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)


In [ ]:
#Train Linear regression model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# -----------------------
# 1. Load dataset
# -----------------------
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")

# -----------------------
# 2. Features and target
# -----------------------
X = train_df.drop("SalePrice", axis=1)
y = np.log1p(train_df["SalePrice"])  # log-transform target

# 80-20 train-test split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------
# 3. Preprocessing
# -----------------------
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# -----------------------
# 4. Pipeline with Linear Regression
# -----------------------
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# -----------------------
# 5. Train the model
# -----------------------
model.fit(X_train, y_train)

# -----------------------
# 6. Predict and evaluate
# -----------------------
y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))



In [ ]:
#Finding R^2, MSE and RMSE

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Predict on validation set
y_pred = model.predict(X_valid)

# R-squared
r2 = r2_score(y_valid, y_pred)

# Mean Squared Error
mse = mean_squared_error(y_valid, y_pred)

# Root Mean Squared Error
rmse = np.sqrt(mse)

print(f"R²: {r2:.4f}")
print(f"MSE (log scale): {mse:.4f}")
print(f"RMSE (log scale): {rmse:.4f}")


In [ ]:
#Feature engineering 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load data
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")

# -----------------------
# Feature Engineering
# -----------------------
train_df["TotalSF"] = train_df["TotalBsmtSF"] + train_df["1stFlrSF"] + train_df["2ndFlrSF"]
train_df["TotalBath"] = train_df["FullBath"] + 0.5*train_df["HalfBath"] + train_df["BsmtFullBath"] + 0.5*train_df["BsmtHalfBath"]
train_df["OverallGrade"] = train_df["OverallQual"] * train_df["OverallCond"]
train_df["TotalPorch"] = train_df["OpenPorchSF"] + train_df["EnclosedPorch"] + train_df["3SsnPorch"] + train_df["ScreenPorch"]
train_df["HasPool"] = (train_df["PoolArea"] > 0).astype(int)
train_df["HasGarage"] = (train_df["GarageArea"] > 0).astype(int)

# -----------------------
# Features and target
# -----------------------
X = train_df.drop("SalePrice", axis=1)
y = np.log1p(train_df["SalePrice"])  # log-transform target

# Train-test split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------
# Preprocessing
# -----------------------
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# -----------------------
# Train Linear Regression
# -----------------------
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)

# -----------------------
# Evaluate
# -----------------------
y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
r2 = r2_score(y_valid, y_pred)

print(f"Linear Regression with new features - RMSE: {rmse:.4f}, R²: {r2:.4f}")


In [ ]:
#Training and comparing advanced models 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# -----------------------
# 1. Load data
# -----------------------
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")

# Features and target
X = train_df.drop("SalePrice", axis=1)
y = np.log1p(train_df["SalePrice"])  # log-transform target

# -----------------------
# 2. Preprocessing
# -----------------------
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# -----------------------
# 3. Models to compare
# -----------------------
models = {
    "NeuralNet": MLPRegressor(hidden_layer_sizes=(100,100), max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_state=42, verbose=0)
}

# -----------------------
# 4. Cross-validation
# -----------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model, X, y):
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = -cross_val_score(pipeline, X, y,
                              scoring="neg_root_mean_squared_error",
                              cv=kf,
                              n_jobs=-1)
    return scores.mean()

# -----------------------
# 5. Train and evaluate all models
# -----------------------
results = {}
for name, mdl in models.items():
    print(f"Training {name}...")
    score = rmse_cv(mdl, X, y)
    results[name] = score
    print(f"{name} CV RMSE: {score:.4f}\n")

# Sort results
results_df = pd.DataFrame.from_dict(results, orient='index', columns=['CV_RMSE']).sort_values('CV_RMSE')
print("Model comparison (lower RMSE is better):")
print(results_df)


In [ ]:
#Producing predictions and creating submission file

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

# -----------------------
# 1. Load datasets
# -----------------------
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")

# -----------------------
# 2. Features and target
# -----------------------
X = train_df.drop("SalePrice", axis=1)
y = np.log1p(train_df["SalePrice"])

# Identify categorical columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# -----------------------
# 3. Fill missing values in categorical columns
# -----------------------
for col in categorical_cols:
    X[col] = X[col].fillna("Missing")
    test_df[col] = test_df[col].fillna("Missing")

# Fill numeric missing values with 0 (optional)
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
for col in numeric_cols:
    X[col] = X[col].fillna(0)
    test_df[col] = test_df[col].fillna(0)

# -----------------------
# 4. Align test dataframe columns with training
# -----------------------
missing_cols = set(X.columns) - set(test_df.columns)
for col in missing_cols:
    test_df[col] = 0

X_test = test_df[X.columns]

# -----------------------
# 5. Train-validation split
# -----------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------
# 6. Train CatBoost
# -----------------------
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='RMSE',
    random_seed=42,
    verbose=100
)

cat_model.fit(
    X_train, y_train,
    cat_features=categorical_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

# -----------------------
# 7. Predict on test set
# -----------------------
test_pred_log = cat_model.predict(X_test)
test_pred = np.expm1(test_pred_log)

# -----------------------
# 8. Create submission
# -----------------------
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_pred
})

submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")
print(submission.head())
